### Using KnowRob in Python



This notebook demonstrates how to use the KnowRob system directly in Python. It includes importing necessary modules, initializing the knowledge base, and executing queries.

### Importing KnowRob Modules

In [1]:
import json
from knowrob import *

First, we import the required modules from KnowRob. The `try-except` block ensures compatibility with different ROS environments, either using the ROS1-specific package or directly loading `knowrob.so`.

In [2]:
InitKnowRob()

[15:18:22.697] [info] [KnowRob] static initialization done.


The `InitKnowRob()` function initializes the KnowRob system, setting up necessary configurations and connections.

### Setting Up Knowledge Base

In [3]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"}
            # {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        # {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrl2",
			"read-only": False
		}
	],
	"reasoner": [
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

[15:18:25.761] [info] Using backend `mongodb` with type `MongoDB`.
[15:18:25.761] [info] [mongodb] connected to mongodb://localhost:27017 (swrl2.triples).
[15:18:26.325] [info] Using queryable backend with id 'mongodb'.
[15:18:26.325] [info] Using persistent backend with id 'mongodb'.
[15:18:26.335] [info] Loading ontology at '/home/malineni/ROS_WS/knowrob/owl/rdf-schema.xml' with version "Tue Jun 11 11:53:20 2024" and origin "rdf-schema".
[15:18:26.354] [info] Loading ontology at '/home/malineni/ROS_WS/knowrob/owl/owl.rdf' with version "Tue Jun 11 11:53:20 2024" and origin "owl".
[15:18:26.363] [info] Loading ontology at '/home/malineni/ROS_WS/knowrob/tests/owl/swrl.owl' with version "Tue Jan 14 12:33:30 2025" and origin "swrl".


This block defines the configuration for the KnowledgeBase, including logging, semantic web prefixes, data sources, and backends. The configuration is then serialized to a JSON string and used to initialize the `KnowledgeBase` instance.

### Submitting a Query

In [4]:
phi1 = QueryParser.parse("swrl_test:hasAncestor(swrl_test:'Fred', ?y)")
# phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'AmericanSlicer', ?y)")
# phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'Mozarella', ?y)")

Here, a query is parsed using the `QueryParser`. The query checks for ancestors of the entity `Lea` within the `swrl_test` namespace.

### Retrieving Query Results

In [5]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

The query formulated in the previous step is submitted to the KnowledgeBase. The results are retrieved as a stream, and a queue is created to handle them.

### Processing Query Results


In [6]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

?y : swrl_test:Rex


This block checks if the result is affirmative (`AnswerYes`) and prints each substitution found in the query result, listing variable bindings.

### Negative Query Result Handling


In [7]:
phi2 = QueryParser.parse("swrl_test:hasSibling(swrl_test:'Ernest', swrl_test:'Fred')")
resultStream = kb.submitQuery(phi2, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult2 = resultQueue.pop_front()
if isinstance(nextResult2, AnswerNo):
    print("result is negative")
else:
    print("result is positive")

result is positive


A second query checks for a specific condition, in this case, whether `Lea` is an ancestor of herself, which is expected to be false. The result is handled accordingly.

### Inconclusive Query Result Handling

In [8]:
phi3 = QueryParser.parse("r(?x, ?y)")
resultStream = kb.submitQuery(phi3, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult3 = resultQueue.pop_front()
if isinstance(nextResult3, AnswerDontKnow):
    print("We can't say if the result is true or false")

We can't say if the result is true or false
[12:30:28.740] [warning] Predicate r(?x, ?y) is neither materialized in the EDB nor defined by a reasoner.



The final example demonstrates handling a situation where the system cannot determine the truth value of the query, resulting in an `AnswerDontKnow` response.

### Ontology Reasoner

In [1]:
import json
from knowrob import *
InitKnowRob()

[15:11:17.748] [info] [KnowRob] static initialization done.


In [2]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"},
            # {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
            {"alias": "nlquery", "uri": "http://knowrob.org/kb/nlquery"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        # {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrl",
			"read-only": False
		}
	],
	"reasoner": [
        {
            "name": "OntoQueryReasoner",
            "type": "OntoQueryReasoner",
            "module": "/home/malineni/ROS_WS/knowrob/tutorials/OntoReasoner.py",
			"data-backend": "mongodb",
        }
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

[15:11:17.780] [info] Using backend `mongodb` with type `MongoDB`.
[15:11:17.780] [info] [mongodb] connected to mongodb://localhost:27017 (swrl.triples).
[15:11:17.785] [info] Using queryable backend with id 'mongodb'.
[15:11:17.785] [info] Using persistent backend with id 'mongodb'.
[15:11:17.827] [info] Using reasoner `NLQueryReasoner` with type `NLQueryReasoner`.
[15:11:17.827] [info] Using goal-driven reasoner with id 'NLQueryReasoner'.


In [3]:
phi1 = QueryParser.parse('nlquery:nlquery("get the child classes of the Locomotion class", ?response)')
phi1

nlquery:nlquery("get the child classes of the Locomotion class", ?response)

In [4]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

[15:11:30.229] [error] flask response text type is <class 'str'>


In [5]:
type(nextResult1)

knowrob.AnswerYes

In [6]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

?response : ['AI Message: ', '  Tool: get_ontology_descendant_classes', "  Tool Args: {'ancestor_class_name': 'Locomotion'}", '  Tool Message: ["Approaching", "Locomotion", "Driving", "Flying", "Swimming", "Walking", "MovingAway"]', 'Tool Message: ["Approaching", "Locomotion", "Driving", "Flying", "Swimming", "Walking", "MovingAway"]', 'AI Message: The child classes of the "Locomotion" class are:\n\n1. Approaching\n2. Driving\n3. Flying\n4. Swimming\n5. Walking\n6. MovingAway']


In [7]:
for bind in nextResult1.substitution():
    variable = bind[1]
    term = bind[2]

In [8]:
print(term)

get the child classes of the Locomotion class


### ActionDesignator Reasoner

In [1]:
import json
from knowrob import *
InitKnowRob()

[16:02:47.677] [info] [KnowRob] static initialization done.


In [2]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"},
            {"alias": "nlquery", "uri": "http://knowrob.org/kb/nlquery"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrlad",
			"read-only": False
		}
	],
	"reasoner": [
        {
            "name": "ADReasoner",
            "type": "ADReasoner",
            "module": "/home/malineni/ROS_WS/knowrob/tutorials/ActionDesignatorReasoner.py",
			"data-backend": "mongodb",
        }
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)
phi1 = QueryParser.parse('nlquery:nlquery("cut the brocolli using the big sharp black knife on the desk", ?response)')
phi1

[16:02:47.795] [info] Using backend `mongodb` with type `MongoDB`.
[16:02:47.795] [info] [mongodb] connected to mongodb://localhost:27017 (swrlad.triples).
[16:02:47.797] [info] Using queryable backend with id 'mongodb'.
[16:02:47.797] [info] Using persistent backend with id 'mongodb'.
[16:02:47.827] [info] Using reasoner `ADReasoner` with type `ADReasoner`.
[16:02:47.827] [info] Using goal-driven reasoner with id 'ADReasoner'.


In [4]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

[16:03:27.607] [error] flask response text type is <class 'str'>


In [5]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

?response : ```lisp
(an action
    (type cutting)
    (object (an object
              (type broccoli)
              (name "fresh-broccoli")
              (properties (size "large")
                          (color "green")
                          (texture "bumpy"))))
    (tool (a tool
            (type knife)
            (name "big-sharp-black-knife")
            (properties (sharpness "very-high")
                        (size "large")
                        (material "steel")
                        (color "black")
                        (weight "heavy")
                        (edge "serrated"))))
    (location (a location
                (type desk)
                (name "kitchen-desk")
                (properties (material "wood")
                            (height 0.8)
                            (accessibility "high")
                            (surface-type "stable"))))
    (goal (for-object (an object
                        (type broccoli)
                        (to b